# Custom CNN 43-Class Traffic Sign Classifier Training

This notebook downloads your custom Kaggle dataset using the Kaggle API (kaggle.json), extracts the raw GTSRB dataset, crops and resizes it to 32x32 pixels using ROI coordinates, splits the dataset, and trains your custom CNN model. This runs entirely in Google Colab or Kaggle without uploading local data.

### Step 1: Upload your Kaggle API credentials (kaggle.json)
Run this cell and upload the `kaggle.json` file downloaded from your Kaggle Account Settings.

In [ ]:
from google.colab import files
files.upload() # Select your kaggle.json file

### Step 2: Configure Kaggle API and Download Dataset
Move your credentials to the correct location and download your custom Kaggle dataset containing the raw zips.

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d hanuma2048/trafficsense-raw-data

### Step 3: Extract Dataset and Clean Up Zips
First, we extract the zip downloaded from Kaggle. Then, we extract the raw `GTSRB_Final_Training_Images.zip` containing the classification dataset into `gtsrb_raw/`. Finally, we clean up the zip files to save space.

In [ ]:
import zipfile
from pathlib import Path

kaggle_zip = Path("trafficsense-raw-data.zip") 
gtsrb_zip = Path("GTSRB_Final_Training_Images.zip")
extract_to = Path("gtsrb_raw")

# Extract Kaggle zip bundle
if kaggle_zip.exists():
    print("Extracting Kaggle dataset bundle...")
    with zipfile.ZipFile(kaggle_zip, 'r') as zip_ref:
        zip_ref.extractall(".")
    print("Kaggle bundle extraction complete.")
    kaggle_zip.unlink()

# Fallback check: If the bundle contained raw zips instead of extracted folders
gtsrb_zip = Path("GTSRB_Final_Training_Images.zip")
if gtsrb_zip.exists():
    print("Found nested GTSRB_Final_Training_Images.zip, extracting...")
    with zipfile.ZipFile(gtsrb_zip, 'r') as zip_ref:
        zip_ref.extractall("gtsrb_raw_extracted")
    gtsrb_zip.unlink()

### Step 4: Preprocess Classification Data (Crop and Resize to 32x32)
We search the entire workspace dynamically for the GTSRB class folders (locating directory containing `00000`) to handle varying zip formats. We parse the CSV file in each folder to crop the exact region of interest (ROI) of each traffic sign, resize to 32x32, convert PPM to JPEG, and split the data into 80% train and 20% validation splits.

In [ ]:
import csv
import random
from PIL import Image
import shutil
from pathlib import Path

# Search the entire workspace dynamically for GTSRB class folders
class_zero_folders = list(Path(".").glob("**/00000"))
if len(class_zero_folders) > 0:
    gtsrb_src = class_zero_folders[0].parent
    print(f"Found GTSRB images directory at: {gtsrb_src}")
else:
    raise FileNotFoundError("Could not locate class directory '00000' anywhere in the workspace.")

cnn_dir = Path("cnn_data")

# Clear existing processed directory if it exists
if cnn_dir.exists():
    shutil.rmtree(cnn_dir)

# Create splits directories for 43 classes
splits = ["train", "validation"]
for split in splits:
    for i in range(43):
        (cnn_dir / split / f"{i:05d}").mkdir(parents=True, exist_ok=True)

print("Processing and cropping images...")
random.seed(42)
for class_id in range(43):
    class_src_dir = gtsrb_src / f"{class_id:05d}"
    csv_file = class_src_dir / f"GT-{class_id:05d}.csv"
    
    if not class_src_dir.exists() or not csv_file.exists():
        continue
        
    records = []
    with open(csv_file, "r") as f:
        reader = csv.reader(f, delimiter=";")
        header = next(reader)
        for row in reader:
            if len(row) < 8: continue
            filename = row[0]
            roi_x1, roi_y1, roi_x2, roi_y2 = map(int, row[3:7])
            records.append((filename, roi_x1, roi_y1, roi_x2, roi_y2))
            
    random.shuffle(records)
    split_idx = int(len(records) * 0.8)
    
    splits_map = {
        "train": records[:split_idx],
        "validation": records[split_idx:]
    }
    
    for split, records_split in splits_map.items():
        for filename, rx1, ry1, rx2, ry2 in records_split:
            ppm_path = class_src_dir / filename
            jpg_filename = Path(filename).stem + ".jpg"
            dst_path = cnn_dir / split / f"{class_id:05d}" / jpg_filename
            
            try:
                with Image.open(ppm_path) as img:
                    # Crop ROI bounding box
                    cropped = img.crop((rx1, ry1, rx2, ry2))
                    # Resize to 32x32
                    resized = cropped.resize((32, 32), Image.Resampling.LANCZOS)
                    resized.convert("RGB").save(dst_path, "JPEG")
            except Exception as e:
                print(f"Error processing {filename}: {e}")
                
print("Preprocessing complete.")

### Step 5: Clean Up Jupyter Checkpoints
Remove hidden `.ipynb_checkpoints` directories inside train and validation folders to prevent Keras from registering them as classes.

In [ ]:
!rm -rf ./cnn_data/train/.ipynb_checkpoints
!rm -rf ./cnn_data/validation/.ipynb_checkpoints
print("Jupyter hidden folders cleaned.")

### Step 6: Setup Data Generators with Training Augmentation

In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_dir = './cnn_data/train'
val_dir = './cnn_data/validation'

train_datagen = ImageDataGenerator(
    rescale=1.0/255.0,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1
)

val_datagen = ImageDataGenerator(rescale=1.0/255.0)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(32, 32),
    batch_size=64,
    class_mode='sparse'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(32, 32),
    batch_size=64,
    class_mode='sparse',
    shuffle=False
)

### Step 7: Define Custom CNN Model Architecture
Construct the model layers. Batch Normalization is excluded to prevent training numeric instability underflow.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def create_cnn_model(input_shape=(32, 32, 3), num_classes=43):
    model = models.Sequential()
    
    # Block 1
    model.add(layers.Input(shape=input_shape))
    model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))
    model.add(layers.Conv2D(32, (3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.25))
    
    # Block 2
    model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.25))
    
    # Block 3
    model.add(layers.Conv2D(128, (3, 3), activation='relu', padding='same'))
    model.add(layers.Conv2D(128, (3, 3), activation='relu', padding='same'))
    model.add(layers.MaxPooling2D((2, 2)))
    model.add(layers.Dropout(0.25))
    
    # Fully Connected Block
    model.add(layers.Flatten())
    model.add(layers.Dense(512, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(num_classes, activation='softmax'))
    
    return model

model = create_cnn_model()
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

### Step 8: Train the Model

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

checkpoint = ModelCheckpoint(
    'cnn_best.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=40,
    callbacks=[checkpoint, early_stopping]
)

### Step 9: Visualize Training curves and Confusion Matrix
We plot the training/validation loss and accuracy, and generate a classification report and confusion matrix.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# 1. Plot Loss and Accuracy Curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='train_accuracy')
plt.plot(history.history['val_accuracy'], label='val_accuracy')
plt.title('CNN Classification Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.title('CNN Classification Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

# 2. Generate Confusion Matrix on Validation Set
print("Generating confusion matrix on validation dataset...")
val_generator.reset()
predictions = model.predict(val_generator)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = val_generator.classes

# Generate labels
class_indices = val_generator.class_indices
target_names = [f"Class {int(k)}" for k in sorted(list(class_indices.keys()))]

print("\nClassification Report:")
print(classification_report(true_classes, predicted_classes, target_names=target_names))

cm = confusion_matrix(true_classes, predicted_classes)
plt.figure(figsize=(16, 12))
sns.heatmap(cm, annot=False, cmap="Blues")
plt.title("CNN Validation Confusion Matrix")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.show()

### Step 10: Download Best Weights File
Run this cell to download the trained `cnn_best.keras` file directly to your local computer.

In [ ]:
from google.colab import files
files.download('cnn_best.keras')